# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id and their fields
print("Record Sets available in the dataset:")
record_sets = list(metadata.record_sets)
for record_set in record_sets:
    print(f"- Record Set ID: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print("  Fields (@id : name):")
    for field in record_set.fields:
        print(f"    - {field.id} : {field.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose the first record set for demonstration (update as needed to another valid ID)
if len(record_sets) == 0:
    raise RuntimeError('No record sets found in the dataset schema.')

record_set_id = record_sets[0].id

# Optionally, extract a list of all record set IDs
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"Columns in record set {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field by its @id from the list above (substitute as appropriate)
df = dataframes[record_set_id]
numeric_field_id = None
# Try to auto-select a numeric field by inspecting dtype

for col in df.columns:
    sample = df[col].dropna().head(20)
    try:
        sample_as_num = pd.to_numeric(sample)
        if (sample_as_num.dtype == float or sample_as_num.dtype == int) and (sample_as_num.count() > 0):
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is None:
    raise RuntimeError("No numeric field found in the sample record set.")

threshold = df[numeric_field_id].astype(float).quantile(0.5)  # median as a threshold

filtered_df = df[df[numeric_field_id].astype(float) > threshold]
print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
) / filtered_df[numeric_field_id].astype(float).std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another field if available
group_field_candidates = [col for col in df.columns if col != numeric_field_id]
group_field = None
# Pick a groupable/categorical field
for col in group_field_candidates:
    if df[col].dtype == object and df[col].nunique() < 30:
        group_field = col
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,5))
sns.histplot(df[numeric_field_id].astype(float), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# If we have a group field, plot group comparisons
if group_field is not None:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*This notebook demonstrated how to load and inspect a Croissant-described dataset using `mlcroissant`, explore its schema and records using `@id` referencing, and conduct a simple exploratory data analysis, including normalization and grouping on available numeric columns. For further, domain-relevant investigation, consult the dataset's [schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and documentation on its fields and data provenance.*